# SJLIFE: paired clinical vs. Apple Watch

243 subjects, each recorded once on a clinical 12-lead system and once on an
Apple Watch during the same visit. Both sources are lead I over a 10 s window,
so the pairing isolates the device.

`filter_cfg == "own"`, rows kept where `status == "ok"` and
`below_min_beats == 0`.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

df = pd.read_csv("../../results/sjlife_full.csv")
df = df[(df["filter_cfg"] == "own")
        & (df["status"] == "ok")
        & (df["below_min_beats"] == 0)].copy()

# QTVI is a log ratio and goes infinite when the RR variance term vanishes
df["qtvi"] = df["qtvi"].replace([np.inf, -np.inf], np.nan)

print("rows:", len(df), "| subjects:", df["subject_id"].nunique())
df.head()

# Overview: SDQT and QTVI per method and source

In [ ]:
overview = (
    df.groupby(["method", "source"])[["sdqt", "qtvi"]]
    .agg(["count", "mean", "median"])
)
overview

# Apple vs. clinical (Spearman rank correlation)

Per method, pivot `subject_id` x `source` and drop subjects missing either side,
so the correlation runs on complete pairs. Rank-based, so it asks whether the
two sources order subjects the same way rather than whether they agree in
level.

In [ ]:
spearman_rows = []
for metric in ["sdqt", "qtvi"]:
    for method in ["i2dsw", "neurokit"]:
        pivot = (
            df[df["method"] == method]
            .pivot_table(index="subject_id", columns="source", values=metric,
                         aggfunc="first")[["apple", "clinical"]]
            .dropna()
        )
        rho, p_value = spearmanr(pivot["apple"], pivot["clinical"])
        spearman_rows.append({
            "metric": metric,
            "method": method,
            "n_pairs": len(pivot),
            "spearman_rho": rho,
            "p_value": p_value,
        })

spearman_results = pd.DataFrame(spearman_rows, columns=[
    "metric", "method", "n_pairs", "spearman_rho", "p_value",
])
spearman_results